In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.models as models
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
from torchvision.datasets import ImageFolder

문제 1 (단답형 주관식)

- Full Freeze (전체 동결)
- Partial Fine-tuning (부분 해제)
- Full Fine-tuning (전체 해제)

이 세 가지 전략이 각각 어떤 데이터셋 상황(예: 데이터가 많은지/적은지, 원본과 유사한지/다른지)에서 가장 적합한지 간략히 서술하시오.

1번 답을 적어 주세요

1) Full Freeze가 적합한 상황: Full Freeze는 데이터가 적고, 새 데이터셋이 사전 학습 데이터셋과 비슷한 경우에 적합하다. 이미 학습된 특징 추출 능력을 그대로 사용하고, 마지막 분류기만 새로 학습하면 되기 때문이다. 예를 들어 ImageNet으로 학습된 모델을 사용해서 비슷한 일반 이미지 분류 문제를 풀 때 적합하다.



2) Partial Fine-tuning이 적합한 상황: Partial Fine-tuning은 데이터가 어느 정도 있고, 새 데이터셋이 기존 사전 학습 데이터셋과 약간 다른 경우에 적합하다.
초반 계층은 일반적인 특징을 잘 추출하므로 고정하고, 뒤쪽 계층 일부만 풀어서 새 데이터에 맞게 조정한다. 즉, 완전히 새로 학습하기에는 데이터가 부족하지만, 기존 특징만 그대로 쓰기에는 부족할 때 사용한다.


3) Full Fine-tuning이 적합한 상황: Full Fine-tuning은 데이터가 많고, 새 데이터셋이 기존 사전 학습 데이터셋과 많이 다른 경우에 적합하다. 모델 전체를 새 데이터에 맞게 다시 조정할 수 있기 때문이다. 다만 모든 계층을 학습하므로 학습 시간이 오래 걸리고, 데이터가 적으면 과적합이 발생할 수 있다.

문제 2 (단답형 주관식)

'Full Fine-tuning(전체 해제)' 전략을 사용할 때, 모델의 모든 계층을 동일한 학습률(Learning Rate)로 학습시키지 않고 계층별로 학습률을 다르게 설정하는 "차등 학습률
(Discriminative Learning Rates)" 기법을 적용했습니다.
이 기법을 사용하는 이유를 '백본(Backbone)'과 '분류기(Classifier)'의 관점에서 서술하시오.

Full Fine-tuning에서는 모델 전체를 학습하지만, 모든 계층을 같은 학습률로 업데이트하면 기존에 잘 학습된 특징이 너무 크게 변할 수 있다.

백본은 이미지의 기본적인 특징을 추출하는 부분이고, 이미 사전 학습을 통해 유용한 특징을 많이 알고 있다. 그래서 백본에는 작은 학습률을 적용해서 기존 지식을 크게 망가뜨리지 않도록 한다.

반면 분류기는 새 데이터셋의 클래스 수와 목적에 맞게 새로 학습해야 하는 부분이다. 따라서 분류기에는 상대적으로 큰 학습률을 적용해서 빠르게 새 문제에 적응하도록 한다.

즉, 차등 학습률은 백본은 천천히 조정하고, 분류기는 빠르게 학습시키기 위해 사용한다.

문제 3 (실습 문제 - 코드 빈칸 채우기)

torchvision.models에서 resnet50 모델을 불러온 뒤, "Full Freeze(전체 동결)" 전략을 적용하기 위해 모델의 모든 파라미터를 고정(freeze)하는 코드입니다.

2개의 빈칸 (# TODO: ...)을 채워 파라미터 고정 로직을 완성하시오.

In [1]:
import torchvision.models as models

# --- 사전 학습된 ResNet-50 모델 로드 (수정 불필요) ---
model = models.resnet50(pretrained=True)
# ------------------------------------------------

print(f"변경 전 (예시: layer1): {model.layer1[0].conv1.weight.requires_grad}")
print(f"변경 전 (예시: fc): {model.fc.weight.requires_grad}")

# TODO: 1. model의 모든 파라미터를 순회(loop)
for param in model.parameters():

    # TODO: 2. 각 파라미터(param)의 requires_grad를 False로 설정하여 고정
    param.requires_grad = False


# --- 결과 확인 (수정 불필요) ---
print(f"\n변경 후 (예시: layer1): {model.layer1[0].conv1.weight.requires_grad}")
print(f"변경 후 (예시: fc): {model.fc.weight.requires_grad}")
# (참고: 이후 fc는 새 레이어로 교체되므로 True가 됩니다)

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 178MB/s] 


변경 전 (예시: layer1): True
변경 전 (예시: fc): True

변경 후 (예시: layer1): False
변경 후 (예시: fc): False


문제 4 (실습 문제 - 코드 빈칸 채우기)

"Full Freeze" 전략을 위해 사전 학습 모델의 마지막 fc 계층을 새로운 분류기(Classifier)로 교체하는 코드입니다.

(ResNet-50 기준)

3개의 빈칸 (# TODO: ...)을 채워 마지막 fc 레이어를 교체하시오.

In [3]:
import torch.nn as nn
import torchvision.models as models

# --- 사전 학습된 ResNet-50 모델 로드 (수정 불필요) ---
model = models.resnet50(pretrained=True)
print(f"원본 FC 레이어:\n{model.fc}\n")
# ------------------------------------------------

# TODO: 1. ResNet-50의 'fc' 계층의 입력 특성 수(in_features) 가져오기
n_features = model.fc.in_features

# TODO: 2. 새로운 출력 클래스 수
num_classes = 5 # (강의 예제와 동일하게 5개로 가정)

# TODO: 3. model.fc를 n_features 입력, num_classes 출력을 갖는 새로운 nn.Linear 계층으로 교체
model.fc = nn.Linear(n_features, num_classes)


# --- 결과 확인 (수정 불필요) ---
print(f"변경된 FC 레이어:\n{model.fc}")
# (참고: 새로 정의된 레이어는 requires_grad가 True입니다)
print(f"변경된 FC 레이어의 학습 여부: {model.fc.weight.requires_grad}")

원본 FC 레이어:
Linear(in_features=2048, out_features=1000, bias=True)

변경된 FC 레이어:
Linear(in_features=2048, out_features=5, bias=True)
변경된 FC 레이어의 학습 여부: True


문제 5 (실습 문제 - 코드 빈칸 채우기)

"Full Freeze(전체 동결)" 전략을 위한 옵티마이저를 정의하는 코드입니다.

이 전략은 새로 교체된 model.fc 계층의 파라미터만 학습해야 합니다.

1개의 빈칸 (# TODO: ...)을 채워 옵티마이저가 model.fc.parameters()만 학습하도록 설정하시오.

In [4]:
import torch.optim as optim
import torch.nn as nn
import torchvision.models as models

# --- 모델 준비 (수정 불필요) ---
model = models.resnet50(pretrained=True)
# (모든 파라미터 고정 가정)
for param in model.parameters():
    param.requires_grad = False
# (분류기 교체)
model.fc = nn.Linear(model.fc.in_features, 5)
base_lr = 0.01
# ---------------------------------

optimizer = optim.SGD(

    # TODO: 1. 학습할 파라미터로 'model.fc.parameters()'만 지정
    model.fc.parameters(),

    lr=base_lr,
    momentum=0.9
)

# --- 결과 확인 (수정 불필요) ---
print("옵티마이저가 학습할 파라미터 그룹:")
for param_group in optimizer.param_groups:
    print(f"Learning Rate: {param_group['lr']}")
    print(f"파라미터 수: {len(param_group['params'])}") # 2개 (weight, bias)

옵티마이저가 학습할 파라미터 그룹:
Learning Rate: 0.01
파라미터 수: 2


문제 6 (실습 문제 - 코드 작성)

"Full Fine-tuning(전체 해제)" 전략을 위해 차등 학습률(Discriminative Learning Rates)을 적용하는 옵티마이저를 정의하는 코드입니다.

[요구사항]

optim.SGD의 파라미터 리스트에 3개의 딕셔너리를 전달하여 다음과 같이 학습률을 차등 적용하시오.

- model.fc 파라미터: 학습률 base_lr (1.0배)
- model.layer4 파라미터: 학습률 base_lr * 0.1 (0.1배)
- 그 외 모든 파라미터: 학습률 base_lr * 0.01 (0.01배)

(힌트: model.parameters()는 제너레이터(generator)이므로 한 번 순회하면 비워집니다. other_params 정의 시 model.fc와 model.layer4의 파라미터를 제외해야 합니다.)

In [5]:
import torch.optim as optim
import torch.nn as nn
import torchvision.models as models

# --- 모델 준비 (수정 불필요) ---
model = models.resnet50(pretrained=True)
model.fc = nn.Linear(model.fc.in_features, 5)
base_lr = 0.01
# ---------------------------------

# TODO: 1. model.fc와 model.layer4의 파라미터 ID 저장
fc_params = set(model.fc.parameters())
layer4_params = set(model.layer4.parameters())

# TODO: 2. fc와 layer4를 제외한 'other_params' 리스트 생성
other_params = [param for param in model.parameters() if
                (param not in fc_params) and (param not in layer4_params)]

# TODO: 3. 차등 학습률을 적용하기 위한 'params_to_optimize' 리스트 정의
params_to_optimize = [
    # (그룹 1: model.fc 파라미터)
    {'params': model.fc.parameters(), 'lr': base_lr},

    # (그룹 2: model.layer4 파라미터)
    {'params': model.layer4.parameters(), 'lr': base_lr * 0.1},

    # (그룹 3: other_params)
    {'params': other_params, 'lr': base_lr * 0.01}
]

# TODO: 4. params_to_optimize를 optim.SGD에 전달
optimizer = optim.SGD(
    params_to_optimize,
    momentum=0.9
)


# --- 결과 확인 (수정 불필요) ---
print("옵티마이저가 학습할 파라미터 그룹:")
for i, param_group in enumerate(optimizer.param_groups):
    print(f"\n[그룹 {i+1}]")
    print(f"Learning Rate: {param_group['lr']:.4f}")
    print(f"파라미터 수: {len(param_group['params'])}")

옵티마이저가 학습할 파라미터 그룹:

[그룹 1]
Learning Rate: 0.0100
파라미터 수: 2

[그룹 2]
Learning Rate: 0.0010
파라미터 수: 30

[그룹 3]
Learning Rate: 0.0001
파라미터 수: 129
